# 👵 Nursing Home Fall Detection - YOLO11 Training (EXP-006: Hybrid Optimization)

Notebook ini digunakan untuk melatih model **YOLO11 Nano** dengan optimasi **EXP-006: The Hybrid** (AdamW + Cosine LR + Extended Patience 30 + Night/IR Augmentation + Class Loss Tuning).

### 📌 Petunjuk Penggunaan di Kaggle:
1. Nyalakan GPU di menu: **Settings -> Accelerator -> GPU T4 x1**.
2. Pastikan dataset sudah ditambahkan via menu: **+ Add Data** -> pilih dataset `fall-detection-dataset` (Path: `/kaggle/input/datasets/rajarahmanaziiz/fall-detection-dataset`).
3. Dataset langsung terbaca dari folder (tidak perlu diekstrak dari .zip).
4. Jalankan cell secara berurutan dari atas ke bawah.

### 1. Install Ultralytics & Verifikasi Akselerasi GPU

In [ ]:
!pip install -q ultralytics

import torch
import yaml
from pathlib import Path
from ultralytics import YOLO

print('PyTorch Version :', torch.__version__)
print('CUDA Available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device Name     :', torch.cuda.get_device_name(0))


### 2. Deteksi Dataset Kaggle (Folder Langsung / Uncompressed)

In [ ]:
# Path utama dataset Kaggle
primary_dataset_path = Path('/kaggle/input/datasets/rajarahmanaziiz/fall-detection-dataset')
fallback_candidates = [
    primary_dataset_path,
    Path('/kaggle/input/fall-detection-dataset'),
    Path('/kaggle/input/urfall-yolo-dataset'),
]

dataset_dir = None

# Cek path utama atau fallback folder
for candidate in fallback_candidates:
    if candidate.exists():
        if (candidate / 'images').exists():
            dataset_dir = candidate
            break
        sub_images = list(candidate.glob('**/images'))
        if sub_images:
            dataset_dir = sub_images[0].parent
            break

if not dataset_dir:
    kaggle_input = Path('/kaggle/input')
    found_images = list(kaggle_input.glob('**/images'))
    if found_images:
        dataset_dir = found_images[0].parent

if dataset_dir:
    print(f'✅ Dataset berhasil ditemukan di: {dataset_dir}')
    print(f'   Train images : {len(list((dataset_dir / "images" / "train").glob("*")))} file')
    print(f'   Val images   : {len(list((dataset_dir / "images" / "val").glob("*")))} file')
    print(f'   Test images  : {len(list((dataset_dir / "images" / "test").glob("*")))} file')
else:
    raise FileNotFoundError('❌ Folder dataset tidak ditemukan di /kaggle/input! Periksa input dataset Anda.')


### 3. Generate data.yaml di Lingkungan Kaggle

In [ ]:
work_dir = Path('/kaggle/working')
kaggle_yaml_path = work_dir / 'data.yaml'

data_yaml_content = {
    'path': str(dataset_dir.resolve()),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 3,
    'names': {
        0: 'normal',
        1: 'transitional',
        2: 'lying_on_ground'
    }
}

with open(kaggle_yaml_path, 'w') as f:
    yaml.dump(data_yaml_content, f, default_flow_style=False)

print(f'✓ data.yaml berhasil dibuat di: {kaggle_yaml_path}')
print('-' * 35)
print(open(kaggle_yaml_path).read())
print('-' * 35)


### 4. Training EXP-006: The Hybrid Optimization

> 🔬 **HIPOTESIS EXP-006:**
> - **Model**: `yolo11n.pt` (terbukti tidak overfit pada 924 sampel).
> - **Optimizer**: `AdamW` + `cos_lr=True` (`lr0=0.001`) untuk kestabilan gradient dan menaikkan Precision.
> - **Patience 30 & Epochs 70**: Mempertahankan Recall rekor tinggi seperti EXP-005 (>= 90%).
> - **Augmentasi Realita Panti**: `hsv_v=0.6` (cahaya minim), `bgr=0.2` (mode infrared CCTV malam), `erasing=0.4` (occlusion selimut/kursi).
> - **Class Weight (`cls=1.0`)**: Mempertajam pemisahan antara postur `lying_on_ground` dan `transitional`.

In [ ]:
print('🚀 Memulai Training EXP-006...')
model = YOLO('yolo11n.pt')

results = model.train(
    data=str(kaggle_yaml_path),
    epochs=70,
    patience=30,
    batch=16,
    imgsz=640,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    cos_lr=True,
    cls=1.0,
    hsv_v=0.6,
    hsv_s=0.7,
    hsv_h=0.015,
    bgr=0.2,
    erasing=0.4,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    close_mosaic=10,
    project='/kaggle/working/models',
    name='exp006_hybrid_adamw',
    exist_ok=True,
    save=True
)

print('\n🎉 Training EXP-006 selesai!')
print(f'Hasil tersimpan di: {results.save_dir}')


### 5. Evaluasi pada Test Set & Cetak Metrik Lengkap

In [ ]:
print('🔍 Menjalankan evaluasi pada Test Set...')
metrics = model.val(data=str(kaggle_yaml_path), split='test')

print('\n' + '='*50)
print('📊 HASIL METRIK TEST SET EXP-006:')
print('='*50)
print(f'• Test mAP50    : {metrics.box.map50 * 100:.2f}%')
print(f'• Test mAP50-95 : {metrics.box.map * 100:.2f}%')
print(f'• Test Precision: {metrics.box.mp * 100:.2f}%')
print(f'• Test Recall   : {metrics.box.mr * 100:.2f}%')
print('='*50)
print('\n📁 Model terbaik kalian tersimpan di:')
print(f'{results.save_dir}/weights/best.pt')
print('Silakan unduh best.pt dan grafik evaluasi dari tab Output Kaggle!')
